# Veille Technologique : Comparaison Multimodale (CLIP vs Baseline)

Ce notebook implémente la preuve de concept (POC) décrite dans le README.
Nous comparons deux approches pour la classification de produits :
1. **Baseline** : Fusion tardive (EfficientNet B0 + BERT)
2. **Challenger** : CLIP (OpenAI)

## Pré-requis
Assurez-vous d'avoir les données :
- `produits_clean.csv` dans le dossier `Notebook/` ou à la racine (ajustez le chemin ci-dessous).
- Un dossier `données_veille_tech` contenant les images.


In [ ]:
#!pip install torch torchvision transformers timm pandas scikit-learn pillow matplotlib tqdm

In [ ]:
# --- Import des bibliothèques --- #
import os
import sys
import pandas as pd
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, adjusted_rand_score

# Bibliothèques Deep Learning
import timm
from transformers import AutoTokenizer, AutoModel, CLIPProcessor, CLIPModel

# --- Configuration des chemins ---
# Ajout du chemin racine pour pouvoir importer `config`
# Le notebook est dans Notebook/, donc la racine est le parent '..'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config.config import config

# Configuration Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilisation du device : {device}")

/Users/bervie/Documents/PC/RAS/Alternance/Openclassrooms/Data Scientist/projet_8_Veille_du_model_de_scoring/ScoringModel/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Utilisation du device : cpu


In [3]:
# --- CONFIGURATION --- #
# Les chemins sont maintenant gérés via le fichier de configuration central
CSV_PATH = os.path.join(config.BASE_DIR, "données_veille_tech/produits_clean.csv")
IMG_DIR = os.path.join(config.BASE_DIR, "données_veille_tech/images")

# Chargement des données
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"Dataset chargé : {df.shape[0]} produits")
    display(df.head())
else:
    print(f"ERREUR : Fichier CSV '{CSV_PATH}' introuvable. Veuillez vérifier le chemin.")

Dataset chargé : 1050 produits


,uniq_id,product_name,description,product_category_tree,image,category_main,description_clean,image_array,image_cnn
0,55b85ea15a1536d46b7190ad6fff8ce7,Elegance Polyester Multicolor Abstract Eyelet ...,Key Features of Elegance Polyester Multicolor ...,"[""Home Furnishing >> Curtains & Accessories >>...",55b85ea15a1536d46b7190ad6fff8ce7.jpg,home furnishing,key feature elegance polyester multicolor abst...,[[206 206 207 ... 171 170 170]\n [207 208 208 ...,[[[-122.876076 -115.98292 -103.18606 ]\n [-1...
1,7b72c92c2f6c40268628ec5f14c6d590,Sathiyas Cotton Bath Towel,Specifications of Sathiyas Cotton Bath Towel (...,"[""Baby Care >> Baby Bath & Skin >> Baby Bath T...",7b72c92c2f6c40268628ec5f14c6d590.jpg,baby care,specification sathiyas cotton bath towel b...,[[255 255 255 ... 255 255 255]\n [255 255 255 ...,[[[-122.68 -115.779 -102.939]\n [-122.68 -1...
2,64d5d4a258243731dc7bbb1eef49ad74,Eurospa Cotton Terry Face Towel Set,Key Features of Eurospa Cotton Terry Face Towe...,"[""Baby Care >> Baby Bath & Skin >> Baby Bath T...",64d5d4a258243731dc7bbb1eef49ad74.jpg,baby care,key feature eurospa cotton terry face towel se...,[[255 255 255 ... 255 255 255]\n [255 255 255 ...,[[[-122.68 -115.779 -102.939]\n [-122.68 -1...
3,d4684dcdc759dd9cdf41504698d737d8,SANTOSH ROYAL FASHION Cotton Printed King size...,Key Features of SANTOSH ROYAL FASHION Cotton P...,"[""Home Furnishing >> Bed Linen >> Bedsheets >>...",d4684dcdc759dd9cdf41504698d737d8.jpg,home furnishing,key feature santosh royal fashion cotton print...,[[255 255 255 ... 255 255 255]\n [255 255 255 ...,[[[-122.68 -115.779 -102.939 ]\n [-1...
4,6325b6870c54cd47be6ebfbffa620ec7,Jaipur Print Cotton Floral King sized Double B...,Key Features of Jaipur Print Cotton Floral Kin...,"[""Home Furnishing >> Bed Linen >> Bedsheets >>...",6325b6870c54cd47be6ebfbffa620ec7.jpg,home furnishing,key feature jaipur print cotton floral king si...,[[161 158 152 ... 221 223 224]\n [155 151 144 ...,[[[-123.58196 -116.69665 -103.88802 ]\n [-1...


## 1. Préparation des Modèles (Baseline)

In [4]:
# 1. EfficientNet (Image)
# On charge un modèle pré-entraîné et on enlève la tête de classification (num_classes=0)
effnet = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
effnet.to(device)
effnet.eval()

# Transformation d'image standard pour EfficientNet
from torchvision import transforms
effnet_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. BERT (Texte)
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_model.to(device)
bert_model.eval()

print("Modèles Baseline chargés.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2629.63it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modèles Baseline chargés.


## 2. Préparation du Modèle Challenger (CLIP)

In [5]:
# CLIP (Image + Texte)
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

print("Modèle CLIP chargé.")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2607.08it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Modèle CLIP chargé.


## 3. Extraction des Features
Nous allons itérer sur le dataset pour extraire les vecteurs caractéristiques.

In [6]:
# selction des colonnes à garder
colonnes_gardees = ['uniq_id', 'image', 'description', 'category_main']
df = df[colonnes_gardees]

In [7]:
# --- Extraction des features --- #
def extract_features(df, img_dir, max_samples=None):
        features_baseline = []
        features_clip = []
        labels = []

        if max_samples:
            df = df.head(max_samples)

        print("Extraction des features en cours...")
        for idx, row in tqdm(df.iterrows(), total=len(df)):
            # Adaptation aux colonnes spécifiques
            img_name = row['image']
            text = row['description']
            label = row['category_main']

            if pd.isna(img_name):
                continue
                
            img_path = os.path.join(img_dir, img_name)
            if not os.path.exists(img_path):
                continue

            try:
                image = Image.open(img_path).convert("RGB")

                # --- BASELINE --- #
                # 1. Image (EfficientNet)
                img_tensor = effnet_transforms(image).unsqueeze(0).to(device)
                with torch.no_grad():
                    eff_emb = effnet(img_tensor).cpu().numpy().flatten()

                # 2. Texte (BERT)
                inputs = bert_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
                with torch.no_grad():
                    bert_out = bert_model(**inputs)
                    bert_emb = bert_out.last_hidden_state[:, 0, :].cpu().numpy().flatten()

                baseline_vec = np.concatenate([eff_emb, bert_emb])
                features_baseline.append(baseline_vec)

                # --- CHALLENGER (CLIP) --- #
                inputs_clip = clip_processor(text=[text], images=image, return_tensors="pt", padding=True, truncation=True).to(device)
                with torch.no_grad():
                    outputs_clip = clip_model(**inputs_clip)
                    clip_img_emb = outputs_clip.image_embeds.cpu().numpy().flatten()
                    clip_txt_emb = outputs_clip.text_embeds.cpu().numpy().flatten()
                
                clip_vec = np.concatenate([clip_img_emb, clip_txt_emb])
                features_clip.append(clip_vec)

                labels.append(label)

            except Exception as e:
                print(f"Erreur sur {img_name}: {e}")
                continue

        return np.array(features_baseline), np.array(features_clip), np.array(labels)

# Lancer l'extraction (ajuster max_samples si besoin)
X_baseline, X_clip, y = extract_features(df, IMG_DIR, max_samples=None)
print(f"Features Baseline shape: {X_baseline.shape}")
print(f"Features CLIP shape: {X_clip.shape}")

Extraction des features en cours...


 64%|██████▍   | 677/1050 [01:51<01:03,  5.88it/s]/Users/bervie/Documents/PC/RAS/Alternance/Openclassrooms/Data Scientist/projet_8_Veille_du_model_de_scoring/ScoringModel/.venv/lib/python3.12/site-packages/PIL/Image.py:3451: DecompressionBombWarning: Image size (93680328 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
100%|██████████| 1050/1050 [02:52<00:00,  6.08it/s]

Features Baseline shape: (1050, 2048)
Features CLIP shape: (1050, 1024)


## 4. Comparaison des Performances

In [8]:
# Split Train/Test
X_base_train, X_base_test, y_train, y_test = train_test_split(X_baseline, y, test_size=0.2, random_state=42)
X_clip_train, X_clip_test, _, _ = train_test_split(X_clip, y, test_size=0.2, random_state=42)

# Entraînement Baseline
clf_baseline = LogisticRegression(max_iter=1000)
clf_baseline.fit(X_base_train, y_train)
y_pred_base = clf_baseline.predict(X_base_test)

# Entraînement CLIP (Linear Probe)
clf_clip = LogisticRegression(max_iter=1000)
clf_clip.fit(X_clip_train, y_train)
y_pred_clip = clf_clip.predict(X_clip_test)

# Résultats 
print("--- RÉSULTATS BASELINE (EfficientNet + BERT) ---")
print(classification_report(y_test, y_pred_base))
acc_base = accuracy_score(y_test, y_pred_base)
ari_base = adjusted_rand_score(y_test, y_pred_base)
print(f"ARI Baseline : {ari_base:.4f}")

print("\n --- RÉSULTATS CHALLENGER (CLIP) ---")
print(classification_report(y_test, y_pred_clip))
acc_clip = accuracy_score(y_test, y_pred_clip)
ari_clip = adjusted_rand_score(y_test, y_pred_clip)
print(f"ARI CLIP : {ari_clip:.4f}")

print(f"\nGain de performance (Accuracy) : {(acc_clip - acc_base) * 100:.2f} points")
print(f"Gain de performance (ARI)      : {(ari_clip - ari_base):.4f}")

--- RÉSULTATS BASELINE (EfficientNet + BERT) ---
                            precision    recall  f1-score   support

                 baby care       0.70      0.78      0.74        27
  beauty and personal care       0.86      0.86      0.86        21
                 computers       0.97      1.00      0.99        38
home decor & festive needs       0.79      0.90      0.84        30
           home furnishing       0.93      0.74      0.83        35
          kitchen & dining       1.00      0.92      0.96        26
                   watches       0.97      1.00      0.99        33

                  accuracy                           0.89       210
                 macro avg       0.89      0.89      0.89       210
              weighted avg       0.90      0.89      0.89       210

ARI Baseline : 0.7821

 --- RÉSULTATS CHALLENGER (CLIP) ---
                            precision    recall  f1-score   support

                 baby care       0.91      0.78      0.84        27
  b

Les résultats démontrent clairement la supériorité de l'approche moderne (CLIP) sur l'approche classique (Baseline), même sur un jeu de données relativement petit.

Interprétation détaillée des résultats :

1. Performance Globale : Une victoire nette de CLIP

- Gain d'Accuracy (+5.24 points) :
Passer de 89% à environ 94.2% d'exactitude est une amélioration majeure.
Dans le monde du Machine Learning, gagner 5 points quand on est déjà à 89% est difficile. Cela prouve que CLIP capture des nuances que la combinaison BERT + EfficientNet rate complètement.

- Gain d'ARI (+0.10) :
L'ARI passe de 0.78 à 0.88.
Cela signifie que les "clusters" formés par CLIP sont beaucoup plus propres. Les produits d'une même catégorie sont mathématiquement plus proches les uns des autres et mieux séparés des autres catégories dans l'espace vectoriel de CLIP.

2. Analyse par Catégorie : Où CLIP fait-il la différence ?
C'est en regardant les classes difficiles que l'on comprend la puissance de CLIP.

A. Le cas "Baby Care" (Soins bébé)

* Baseline : Précision médiocre (0.70).
Interprétation : Le modèle classique se trompe souvent quand il prédit "Baby Care". Il confond probablement ces produits avec "Beauty and personal care" (shampoings, lotions) car visuellement et textuellement, c'est proche.

* CLIP : Précision excellente (0.91).
Interprétation : CLIP comprend le contexte sémantique. Il sait distinguer une "lotion pour bébé" d'une "lotion pour adulte" grâce à sa compréhension conjointe image/texte.

B. Le cas "Home Furnishing" (Ameublement)

* Baseline : Rappel faible (0.74).
Interprétation : Le modèle classique "rate" plus d'un quart des meubles (26% de Faux Négatifs). Il les classe sans doute ailleurs (peut-être en "Home Decor").

* CLIP : Rappel excellent (0.91).
Interprétation : CLIP récupère presque tous les meubles. Il fait bien la distinction subtile entre un objet de décoration et un meuble, là où la fusion tardive échoue.

C. Les classes faciles (Computers, Watches)

* Les deux modèles excellent (scores > 0.97).
* Ces objets sont visuellement très distincts et ont un vocabulaire spécifique ("RAM", "Dial", "Strap"). Ici, la technologie avancée de CLIP n'est pas indispensable, mais elle atteint la perfection (1.00 partout pour Computers).

##### Conclusion :

- La Fusion Tardive BERT + EfficientNet (Baseline) a ses limites : Coller un vecteur image (EfficientNet) à un vecteur texte (BERT) ne suffit pas toujours à capturer le lien entre les deux.
- L'Espace Multimodal (CLIP) est supérieur : CLIP a été entraîné pour comprendre que l'image d'un berceau et le mot "lit" vont ensemble. Cette "synergie" native explique pourquoi il commet moins d'erreurs de confusion entre des catégories proches (comme Décoration vs Ameublement).

Verdict : L'intégration d'une technologie comme CLIP apporte une réelle valeur ajoutée pour automatiser la classification de produits, réduisant significativement le besoin de vérification humaine (moins d'erreurs).